# 02: Choose where your circuit runs

You will run one circuit on a CPU, then on a CUDA GPU if one is available.
This notebook is independent of notebook 01: it creates its own circuit.

## 1. Build the circuit

You already know these gates. Keep the program the same while changing the execution device.

In [ ]:
import flagquantum as fq
import torch

circuit = fq.Circuit(2)
circuit.h(0)
circuit.cx(0, 1)


## 2. Choose the CPU explicitly

`ExecutionOptions` tells FlagQuantum how to run the program. Here we ask for a statevector calculation on the CPU.

In [ ]:
cpu_options = fq.ExecutionOptions(device="cpu", mode="statevector")
cpu_result = fq.run(circuit, options=cpu_options)
cpu_state = cpu_result.to_statevector().reshape(-1)
print("Device:", cpu_state.device)
print("Number format:", cpu_state.dtype)
print("Probabilities:", cpu_state.abs().square().tolist())


## 3. Check for a GPU

`cuda:0` means the first CUDA device visible to this process. Finding a GPU here means it is already available to this environment; it does not request a new cloud allocation.

In [ ]:
has_gpu = torch.cuda.is_available()
print("CUDA available:", has_gpu)
if has_gpu:
    print("GPU:", torch.cuda.get_device_name(0))


## 4. Run and compare

Change the device option, execute the same circuit, and copy the small result to the CPU for comparison.
If no GPU is available, this cell says so and the exercise ends with the CPU result.

In [ ]:
if has_gpu:
    gpu_options = fq.ExecutionOptions(device="cuda:0", mode="statevector")
    gpu_result = fq.run(circuit, options=gpu_options)
    gpu_state = gpu_result.to_statevector().reshape(-1)
    assert gpu_state.device.type == "cuda"
    torch.testing.assert_close(gpu_state.cpu(), cpu_state, atol=1e-6, rtol=1e-6)
    print("GPU result matches CPU:", gpu_state.device)
else:
    print("GPU step skipped: this environment has no available CUDA device.")


## Think about scale

A two-qubit circuit is too small to demonstrate a useful GPU speedup. Device startup can cost more than the calculation.
For a timing experiment, first warm up the workload, then synchronize CUDA before starting and stopping the timer.

A statevector needs one complex amplitude per bit string. How many entries would you need for 3 qubits? For 20?
Why does adding just one qubit double the storage?